# Phase A0b — Balance check + adaptive rebalance

Reads `confirmed_seeds.json` from A0, computes per-(race × gender) balance against `target_per_cell`, and if any cell is underfilled, iteratively oversamples + re-prefilters until balanced (or max iterations hit). Writes `confirmed_seeds_balanced.json` for Stage A to consume.


In [ ]:
%pip install --force-reinstall --no-deps mediapipe==0.10.20
%pip install ftfy regex torchsde
%pip install /Workspace/Users/alenj00@safeway.com/counterfactual-audit-pipeline

In [ ]:
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text('config_path', '/Workspace/Users/alenj00@safeway.com/counterfactual-audit-pipeline/configs/full.yaml')
dbutils.widgets.text('confirmed_seeds_file', '/Volumes/ds_work/alenj00/cap_cache/runs/full/seed_filter/confirmed_seeds.json')
dbutils.widgets.text('output_dir', '/Volumes/ds_work/alenj00/cap_cache/runs/full/seed_filter')
dbutils.widgets.text('target_per_cell', '14')
dbutils.widgets.text('max_iterations', '5')
dbutils.widgets.text('extra_per_iteration', '50')
CONFIG = dbutils.widgets.get('config_path')
CONFIRMED = dbutils.widgets.get('confirmed_seeds_file')
OUTPUT = dbutils.widgets.get('output_dir')
TARGET = dbutils.widgets.get('target_per_cell')
MAX_IT = dbutils.widgets.get('max_iterations')
EXTRA = dbutils.widgets.get('extra_per_iteration')

In [ ]:
import os
from pathlib import Path
import sys
os.environ['HF_HOME'] = '/local_disk0/hf_cache'
os.environ['HUGGINGFACE_HUB_CACHE'] = '/local_disk0/hf_cache'
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['TORCH_EXTENSIONS_DIR'] = '/dev/shm/torch_extensions'
os.environ['TMPDIR'] = '/dev/shm/tmp'
Path('/dev/shm/torch_extensions').mkdir(parents=True, exist_ok=True)
Path('/dev/shm/tmp').mkdir(parents=True, exist_ok=True)
HF_TOKEN = dbutils.secrets.get(scope='cap-secrets', key='hf-token')
os.environ['HF_TOKEN'] = HF_TOKEN
PULID_SRC = '/Volumes/ds_work/alenj00/cap_cache/pulid_src'
if PULID_SRC not in sys.path: sys.path.insert(0, PULID_SRC)

In [ ]:
from click.testing import CliRunner
from cap.cli.balance_seeds import main as balance_cli
result = CliRunner().invoke(balance_cli, [
  '--config', CONFIG,
  '--confirmed-seeds-file', CONFIRMED,
  '--output-dir', OUTPUT,
  '--target-per-cell', TARGET,
  '--max-iterations', MAX_IT,
  '--extra-per-iteration', EXTRA,
  '--rebalance',
  '--cache-dir', '/local_disk0/hf_cache',
  '--pulid-src', PULID_SRC,
  '--hf-token', HF_TOKEN,
], catch_exceptions=False, standalone_mode=False)
print(result.output[-2500:])

In [ ]:
import json
from pathlib import Path
out = Path(OUTPUT) / 'confirmed_seeds_balanced.json'
ids = json.loads(out.read_text())
msg = f'Balanced confirmed seeds: {len(ids)} IDs → {out}'
print(msg)
dbutils.notebook.exit(msg)